In [17]:
import pandas as pd

In [18]:
pd.set_option('display.max_columns',40)
pd.set_option('display.float_format','{:2f}'.format)

In [19]:
df = pd.read_csv('initial_data.csv', encoding='utf-8')

our_columns = ['DR_Dat','DR_NChk','DR_NDrugs','DR_Kol','DR_CZak','DR_CRoz','DR_SDisc','DR_Suppl']
df = df[our_columns]
df.columns = ['dt','receipt_num','name','quantity','purchase','retail','disc','supplier']

In [20]:
cols = ['purchase', 'quantity', 'retail', 'disc']
df[cols] = df[cols].replace({'\xa0': '', ',': '.'}, regex=True).astype(float)

In [21]:
# ABC-analysis by quantity
import numpy as np

df_gr_1 = (
    df.groupby('name')['quantity']
    .sum().reset_index()
    .sort_values('quantity', ascending=False)
)
df_gr_1['share'] = (
    (df_gr_1['quantity']/sum(df_gr_1['quantity']))*100
    ).sort_values(ascending=False)

df_gr_1['run_total'] = df_gr_1['share'].cumsum()
df_gr_1['ABC_qty'] = np.where(df_gr_1['run_total']<80,'A', np.where(df_gr_1['run_total']<95,'B','C'))
df_gr_1 = df_gr_1[['name','ABC_qty']]


In [22]:
# ABC-analysis by sum of retail sales
df2 = df.copy()
df2['clear'] = (df2['retail']*df2['quantity'])-df['disc']

df_gr_2 = (
    df2.groupby('name')['clear']
    .sum().reset_index()
    .sort_values('clear', ascending=False)
)
df_gr_2['share'] = (
    (df_gr_2['clear']/sum(df_gr_2['clear']))*100
    ).sort_values(ascending=False)

df_gr_2['run_total'] = df_gr_2['share'].cumsum()

df_gr_2['ABC_sales'] = np.where(df_gr_2['run_total']<80,'A', np.where(df_gr_2['run_total']<95,'B','C'))
df_gr_2 = df_gr_2[['name','ABC_sales']]

In [23]:
# ABC_analysis by monthly turnover
df3 = df.copy()

df_gr_3 = (
    ((df3.groupby('name')['quantity']
    .sum()/5)*30)
    .reset_index()
    .sort_values('quantity', ascending=False)
            )
df_gr_3['share'] = (
    (df_gr_3['quantity']/sum(df_gr_3['quantity']))*100
    ).sort_values(ascending=False)

df_gr_3['run_total'] = df_gr_3['share'].cumsum()

df_gr_3['ABC_sales'] = np.where(df_gr_3['run_total']<80,'A', np.where(df_gr_3['run_total']<95,'B','C'))
df_gr_3 = df_gr_3[['name','ABC_sales']]

In [28]:
dff = pd.merge(df_gr_1, df_gr_2, on='name', how='outer')
dff = pd.merge(dff, df_gr_3, on='name', how='outer').sort_values('name', ascending=True)
dff.head(40)

,name,ABC_qty,ABC_sales_x,ABC_sales_y
0,"911-ВЕНОЛГОН ГЕЛЬ Д/НОГ ПРИ ТЯЖЕСТИ,БОЛИ,ОТЕКА...",B,C,B
1,911-ГЕЛЬ-БАЛЬЗАМ Д/НОГ КОНСКИЙ КАШТАН 100МЛ. ТУБА,B,C,B
2,911-ГЕЛЬ-БАЛЬЗАМ Д/СУСТАВОВ РАЗОГР. ОКОПНИК+МУ...,B,C,B
3,L-ТИРОКСИН 100МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/,B,B,B
4,L-ТИРОКСИН 150МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/,B,B,B
5,L-ТИРОКСИН 50МКГ. №50 ТАБ. /БЕРЛИН ХЕМИ/,A,B,A
6,L-ТИРОКСИН 75МКГ. №100 ТАБ. /БЕРЛИН ХЕМИ/,A,A,A
7,NF ВАТА ХИРУРГ. СТЕР. 100Г. /НЬЮФАРМ/,B,C,B
8,АДЖИСЕПТ МЕД+ЛИМОН №24 ТАБ. Д/РАСС.,B,B,B
9,АДЖИСЕПТ МЕНТОЛ+ЭВКАЛИПТ №24 ТАБ. Д/РАСС.,B,B,B
